# LLM Fine-Tuning — Ministral 3B QLoRA
### Customer Support QA · Free Colab T4

**Before running:**  
Runtime → Change runtime type → **T4 GPU** → Save

---
## Step-by-step
| Cell | What it does |
|------|--------------|
| 1 | Check GPU |
| 2 | Mount Google Drive |
| 3 | Unzip project from Drive |
| 4 | Install Python packages |
| 5 | Verify dataset files |
| 6 | **Run QLoRA training** (~60 mins) |
| 7 | Save model adapters to Drive |
| 8 | Run evaluation (before vs after) |
| 9 | Side-by-side inference demo |

---
## Cell 1 — Check GPU
Must show T4. If not: Runtime → Change runtime type → T4 GPU

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {name}')
    print(f'✅ VRAM: {vram:.1f} GB')
    if vram < 10:
        print('⚠️  Less than 10GB VRAM — training may OOM. Switch to T4.')
else:
    print('❌ No GPU found.')
    print('   Go to Runtime → Change runtime type → T4 GPU')

---
## Cell 2 — Mount Google Drive
Your zip file must be at: `My Drive/llm-finetuning.zip`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted at /content/drive')

---
## Cell 3 — Unzip Project
Looks for `llm-finetuning.zip` in your Drive root (`My Drive`).

> **If your zip is in a subfolder**, change `ZIP_PATH` below.

In [ ]:
import os, zipfile, shutil

ZIP_PATH    = '/content/drive/MyDrive/llm-finetuning.zip'
EXTRACT_DIR = '/content/llm-finetuning'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f'Zip not found at {ZIP_PATH}\n'
        f'Upload llm-finetuning.zip to Google Drive root (My Drive) first.'
    )

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')

# Handle nested folder
nested = os.path.join(EXTRACT_DIR, 'llm-finetuning')
if os.path.exists(nested):
    for item in os.listdir(nested):
        shutil.move(os.path.join(nested, item), EXTRACT_DIR)
    os.rmdir(nested)

os.chdir(EXTRACT_DIR)
print(f'✅ Unzipped to {EXTRACT_DIR}')
print(f'✅ Working dir: {os.getcwd()}')
print('\nContents:')
for f in sorted(os.listdir('.')):
    print(f'  {f}')

---
## Cell 4 — Install Packages
Takes ~3 mins. Uses latest compatible versions for Colab Python 3.12 + triton.

In [ ]:
# Upgrade bitsandbytes first — older versions break on Colab's triton
!pip install -q -U bitsandbytes
!pip install -q \
    transformers \
    peft \
    trl \
    datasets \
    accelerate \
    rouge-score \
    nltk \
    pyyaml \
    scipy
print('✅ Packages installed')

# Verify bitsandbytes CUDA works
import bitsandbytes as bnb
print(f'✅ bitsandbytes version: {bnb.__version__}')

---
## Cell 5 — Verify Dataset
Dataset was created on your laptop and included in the zip.
If files are missing, this cell re-creates them.

In [ ]:
import os, json

splits = {
    'train': 'data/processed/train.jsonl',
    'val':   'data/processed/val.jsonl',
    'test':  'data/processed/test.jsonl',
}

missing = [s for s, p in splits.items() if not os.path.exists(p)]

if missing:
    print(f'Missing splits: {missing} — re-creating dataset...')
    !python data/create_dataset.py
else:
    for split, path in splits.items():
        count = sum(1 for _ in open(path))
        print(f'✅ {split}: {count} pairs')

    with open('data/processed/train.jsonl') as f:
        sample = json.loads(f.readline())
    print(f'\nSample Q: {sample["instruction"][:80]}...')
    print(f'Sample A: {sample["output"][:80]}...')

---
## Cell 6 — QLoRA Fine-Tuning
**~60 mins on T4.** Do not close the tab.

Watch the loss drop:
```
Step  10 | Loss: ~2.3
Step 100 | Loss: ~1.1
Step 300 | Loss: ~0.7   ← converging
```
Loss going down = model learning ✅

In [ ]:
!python training/finetune.py

---
## Cell 7 — Save Model Adapters to Drive
Copies `outputs/model/` (~50MB LoRA adapters) to Drive.
Survives Colab session disconnect.

Saved to: `My Drive/llm-finetuning-outputs/model/`

In [ ]:
import os, shutil

SRC  = '/content/llm-finetuning/outputs/model'
DEST = '/content/drive/MyDrive/llm-finetuning-outputs/model'

if not os.path.exists(SRC) or not os.listdir(SRC):
    print('❌ No adapter files found — re-run Cell 6')
else:
    os.makedirs(DEST, exist_ok=True)
    for fname in os.listdir(SRC):
        shutil.copy2(os.path.join(SRC, fname), os.path.join(DEST, fname))
        size_mb = os.path.getsize(os.path.join(DEST, fname)) / 1e6
        print(f'✅ {fname}  ({size_mb:.1f} MB)')
    print(f'\n✅ Saved to Drive: {DEST}')
    print('   Download → place at llm-finetuning/outputs/model/ on laptop')

---
## Cell 8 — Evaluation: Before vs After
Loads base + fine-tuned on 50 test questions. Prints BLEU + ROUGE table.

In [ ]:
!python evaluation/evaluate.py

In [ ]:
import os, shutil

SRC  = '/content/llm-finetuning/outputs/results/evaluation.csv'
DEST = '/content/drive/MyDrive/llm-finetuning-outputs/results/'

if os.path.exists(SRC):
    os.makedirs(DEST, exist_ok=True)
    shutil.copy2(SRC, DEST)
    print('✅ evaluation.csv saved to Drive')
else:
    print('❌ evaluation.csv not found — run Cell 8 first')

---
## Cell 9 — Side-by-Side Inference Demo

In [ ]:
import sys
sys.path.insert(0, '/content/llm-finetuning')
from inference.run_model import load_base_model, load_finetuned_model, generate_response

QUESTION = 'What is your return policy?'

print('Loading base model...')
base_model, base_tok = load_base_model()
print('Loading fine-tuned model...')
ft_model, ft_tok = load_finetuned_model()

print(f'\nQuestion: {QUESTION}')
print('\n' + '='*60)
print('BASE Ministral 3B (no fine-tuning)')
print('='*60)
print(generate_response(base_model, base_tok, QUESTION))
print('\n' + '='*60)
print('FINE-TUNED Ministral 3B (QLoRA)')
print('='*60)
print(generate_response(ft_model, ft_tok, QUESTION))

---
## Done ✅

1. **Download from Drive** → `My Drive/llm-finetuning-outputs/`
   - `model/` → `llm-finetuning/outputs/model/` on laptop
   - `results/evaluation.csv` → `llm-finetuning/outputs/results/`

2. **Run Streamlit demo:**
   ```bash
   streamlit run app.py
   ```

3. Push to GitHub.